# NpuKit — int8 matmul on PYNQ-Z2

Loads `npukit.bit` (+ `.hwh` for DMA) and runs visible cases:

1. **Classic 8x8** — prints **A**, **B**, **C_npu**, **C_ref** for each case
2. **Tiled** — 16x16 demo + larger MxKxN, with a **tiling plan** (what K means)

Math write-up: `docs/tiling.md` in the repo.

Keep `npukit.bit`, `npukit.hwh`, and `npukit_matmul.py` beside this notebook.
After **Run All**, save the notebook so the matrix dumps stay in the file.

In [ ]:
import importlib
import sys

BIT = "/home/xilinx/jupyter_notebooks/npukit.bit"
sys.path.insert(0, "/home/xilinx/jupyter_notebooks")

import npukit_matmul as nk

importlib.reload(nk)
nk.VERBOSE = True  # print A/B/C + tiling for every case

mmio, transport = nk.open_device(BIT)
ident = mmio.read(nk.REG_ID)
assert ident == nk.ID_MAGIC, f"BAD ID 0x{ident:08X}"
print(
    f"ID OK version=0x{mmio.read(nk.REG_VERSION):08X} "
    f"N={mmio.read(nk.REG_N)} transport={type(transport).__name__}"
)

## What tiling means (no FPGA run)

For `C = A @ B` with shapes `A[M x K]`, `B[K x N]`:

- **K** is the inner dimension (dot-product length).
- Hardware only multiplies **8x8** tiles; the host walks spatial blocks of C and, for each block, sums partial products over chunks of K.

In [ ]:
print("Example plan for 16x16x16:")
print(nk.describe_tiling(16, 16, 16))
print()
print("Example plan for 32x32x32:")
print(nk.describe_tiling(32, 32, 32))

## Classic 8x8 suite

Each case prints A, B, FPGA result `C_npu`, and NumPy `C_ref`.

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
classic = nk.classic_8x8_cases(rng)
p0, t0 = nk.run_suite(mmio, transport, classic, verbose=True)
print(f"\nclassic: {p0}/{t0} PASS")
assert p0 == t0

## Tiled suite

Includes a readable **16x16** demo (A = row constants, B = I) plus a random **32x32x32**.

In [ ]:
M, K, Ndim = 32, 32, 32  # must be multiples of 8
tiled = nk.tiled_cases(M, K, Ndim, rng)
p1, t1 = nk.run_suite(mmio, transport, tiled, verbose=True)
print(f"\ntiled: {p1}/{t1} PASS")
assert p1 == t1
print(f"\nall: {p0 + p1}/{t0 + t1} PASS")

## Optional: full CLI re-run

Reloads the bitstream and prints the same verbose suite.

In [ ]:
%run /home/xilinx/jupyter_notebooks/npukit_matmul.py /home/xilinx/jupyter_notebooks/npukit.bit 32 32 32